# 01 · Data audit: IEEE-CIS transactions

**Goal:** understand the data before modelling, and decide how to split it.

**Questions this notebook answers**
1. Is fraud stable over time, and does a time-based split make sense?
2. How do fraud amounts differ from genuine ones?
3. Which products, card types and email domains carry the most risk?
4. Is missing data a signal?
5. Does the rebuilt customer ID behave like a real customer ID?

Run `python scripts/build_master.py` first.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.uid import uid_check

MASTER = Path(os.environ.get("MASTER", ROOT / "data/processed/master.parquet"))
df = pd.read_parquet(MASTER)
plt.rcParams.update({"figure.figsize": (9, 3.5), "axes.spines.top": False, "axes.spines.right": False})
print(f"{len(df):,} transactions, fraud rate {df.isFraud.mean():.2%}")

## 1. Fraud over time

Fraud patterns drift, so the model is trained on early months and tested on later ones (never a random split).
Shaded areas: validation (month 5) and out-of-time test (month 6).

In [ ]:
daily = df.groupby("day").agg(n=("isFraud", "size"), fraud=("isFraud", "mean"))
fig, ax = plt.subplots()
ax.plot(daily.index, daily.fraud.rolling(7, min_periods=1).mean() * 100, color="#2a6fdb")
ax.axvspan(120, 150, color="grey", alpha=0.12, label="validation")
ax.axvspan(150, daily.index.max() + 1, color="grey", alpha=0.25, label="out-of-time")
ax.set(xlabel="day", ylabel="fraud rate % (7-day average)", title="Fraud rate over time")
ax.legend(frameon=False)
plt.show()
df.groupby("split", observed=True)["isFraud"].agg(["size", "mean"]).rename(columns={"size": "transactions", "mean": "fraud_rate"})

## 2. Amounts: fraud vs genuine

In [ ]:
fig, ax = plt.subplots()
for label, colour in [(0, "#9aa5b1"), (1, "#d64545")]:
    ax.hist(np.log10(df.loc[df.isFraud == label, "TransactionAmt"]), bins=60, density=True,
            alpha=0.6, color=colour, label="fraud" if label else "genuine")
ax.set(xlabel="log10(amount)", ylabel="density", title="Transaction amount by label")
ax.legend(frameon=False)
plt.show()
value_share = (df.TransactionAmt * df.isFraud).sum() / df.TransactionAmt.sum()
print(f"Fraud is {df.isFraud.mean():.2%} of transactions but {value_share:.2%} of value")

## 3. Where the risk sits

In [ ]:
def risk_table(col, top=10):
    t = df.groupby(col, observed=True)["isFraud"].agg(transactions="size", fraud_rate="mean")
    return t.sort_values("transactions", ascending=False).head(top).style.format({"fraud_rate": "{:.2%}"})

display(risk_table("ProductCD"))
display(risk_table("card6"))
display(risk_table("P_emaildomain"))

## 4. Missing data is a signal

In fraud data, *whether* a field is filled in often says something (e.g. no device data, no address).

In [ ]:
checks = {
    "has identity data": df.has_identity == 1,
    "addr1 present": df.addr1.notna(),
    "P_emaildomain present": df.P_emaildomain.notna(),
    "D1 present": df.D1.notna(),
}
pd.DataFrame({name: {"share of transactions": m.mean(), "fraud rate if present": df.loc[m, "isFraud"].mean(),
                     "fraud rate if missing": df.loc[~m, "isFraud"].mean()} for name, m in checks.items()}).T.style.format("{:.2%}")

In [ ]:
# V columns come in blocks that are missing together: group them by their missing-value count.
v_cols = [c for c in df.columns if c.startswith("V") and c[1:].isdigit()]
groups = df[v_cols].isna().sum().reset_index().groupby(0)["index"].apply(list)
print(f"{len(v_cols)} V columns fall into {len(groups)} missing-value groups (used in Step 3 to reduce them)")

## 5. Rebuilt customer ID

If `uid` really identifies customers, fraud should cluster: once a customer is flagged, their later transactions are usually labelled fraud too.

In [ ]:
stats = uid_check(df)
display(pd.Series(stats, name="value").to_frame())
per_uid = df[df.uid_complete == 1].groupby("uid", observed=True).size()
per_uid.clip(upper=30).plot.hist(bins=30, title="Transactions per customer (capped at 30)", color="#2a6fdb")
plt.show()

## Findings (write these yourself after running on the real data)

- **Time:** ...
- **Amounts:** ...
- **Risky segments:** ...
- **Missingness:** ...
- **Customer ID:** ...
- **Decision:** train on days 1-119, tune on 120-149, test once on 150+.